In [ ]:
import os
import json
import re
from datetime import datetime, timedelta
import pandas as pd
import traceback # Hata ayıklama için eklendi

class EventMatcher:
    """Excel dosyasından ve klasör isimlerinden event bilgilerini okur"""
    
    def __init__(self, excel_path, mseed_folders_path=None):
        self.excel_path = excel_path
        self.mseed_folders_path = mseed_folders_path
        self.events_info = []
        self.folder_events_info = []
        
    def parse_excel(self):
        """Excel dosyasından event bilgilerini çıkar"""
        print(f"\n📂 Excel dosyası okunuyor: {self.excel_path}")
        
        if not os.path.exists(self.excel_path):
            print(f"⚠️ Excel dosyası bulunamadı: {self.excel_path}")
            return []
        
        try:
            # Excel'i oku (Date kolonunu string olarak oku, sonra parse edeceğiz)
            df = pd.read_excel(self.excel_path)
            
            print(f"✅ Excel okundu: {len(df)} satır")
            print(f"   Kolonlar: {list(df.columns)}")
            
            # İlk Date değerini kontrol et
            if len(df) > 0 and 'Date' in df.columns:
                first_date_raw = df['Date'].iloc[0]
                print(f"   İlk tarih (raw): {first_date_raw} (tip: {type(first_date_raw)})")
            
            # Her satırı işle
            for idx, row in df.iterrows():
                try:
                    # Date kolonunu parse et
                    date_val = row['Date']
                    
                    # Pandas Timestamp ise datetime'a çevir
                    if hasattr(date_val, 'to_pydatetime'):
                        event_datetime = date_val.to_pydatetime()
                    elif pd.notna(date_val):
                        # String ise Türk/Avrupa formatında (DD/MM/YYYY) parse et
                        if isinstance(date_val, str):
                            # Formatlar: DD/MM/YYYY HH:MM:SS veya DD/MM/YYYY
                            event_datetime = pd.to_datetime(date_val, format='%d/%m/%Y %H:%M:%S', errors='coerce')
                            if pd.isna(event_datetime):
                                event_datetime = pd.to_datetime(date_val, format='%d/%m/%Y', errors='coerce')
                            if pd.isna(event_datetime):
                                # Fallback: pandas'ın otomatik parse'ını dene (dayfirst=True ile)
                                event_datetime = pd.to_datetime(date_val, dayfirst=True, errors='coerce')
                        else:
                            # Sayısal veya başka format
                            event_datetime = pd.to_datetime(date_val, dayfirst=True, errors='coerce')
                        
                        if hasattr(event_datetime, 'to_pydatetime'):
                            event_datetime = event_datetime.to_pydatetime()
                    else:
                        continue
                    
                    # EventID
                    event_id = str(int(row['EventID'])) if pd.notna(row['EventID']) else None
                    
                    # Magnitude
                    magnitude = float(row['Magnitude']) if pd.notna(row['Magnitude']) else None
                    
                    # Type (magnitude type)
                    mag_type = str(row['Type']).strip() if pd.notna(row['Type']) else None
                    
                    if event_id and event_datetime:
                        # Sadece tarih kısmını al (yıl, ay, gün)
                        # Saat bilgisini de sakla ama eşleştirmede kullanma
                        event_info = {
                            'event_id': event_id,
                            'datetime': event_datetime,
                            'year': event_datetime.year,
                            'month': event_datetime.month,
                            'day': event_datetime.day,
                            'hour': event_datetime.hour,
                            'minute': event_datetime.minute,
                            'second': event_datetime.second,
                            'magnitude': magnitude,
                            'magnitude_type': mag_type,
                            'location': str(row['Location']) if pd.notna(row['Location']) else None,
                            'date_string': f"{event_datetime.year:04d}-{event_datetime.month:02d}-{event_datetime.day:02d}"
                        }
                        
                        self.events_info.append(event_info)
                        
                        # İlk 3 satırı debug için göster
                        if idx < 3:
                            print(f"     Satır {idx+1}: {event_info['date_string']} {event_datetime.hour:02d}:{event_datetime.minute:02d}:{event_datetime.second:02d} - ID={event_id}, Mag={magnitude}")
                
                except Exception as e:
                    # Hatalı satırları atla
                    continue
            
            print(f"✅ {len(self.events_info)} event yüklendi")
            
            # Tarih gruplarını göster
            date_groups = {}
            for event in self.events_info:
                date_key = event['date_string']
                if date_key not in date_groups:
                    date_groups[date_key] = []
                date_groups[date_key].append(event['event_id'])
            
            print(f"\n   📅 Tarih grupları: {len(date_groups)} farklı gün")
            
            # Excel'deki tüm tarihleri listele (küçük bir örnek)
            all_dates = sorted(date_groups.keys())
            if len(all_dates) > 0:
                print(f"   İlk tarih: {all_dates[0]}")
                print(f"   Son tarih: {all_dates[-1]}")
                print(f"   Örnek tarihler: {', '.join(all_dates[:10])}")
            
            # Birden fazla deprem olan günleri göster
            multi_event_days = {k: v for k, v in date_groups.items() if len(v) > 1}
            if multi_event_days:
                print(f"\n   ⚠️ Birden fazla deprem olan günler: {len(multi_event_days)}")
                for date_str, event_ids in sorted(multi_event_days.items())[:5]:
                    print(f"     {date_str}: {len(event_ids)} deprem (IDs: {', '.join(event_ids[:3])}...)")
            
            return self.events_info
            
        except Exception as e:
            print(f"❌ Excel okuma hatası: {e}")
            traceback.print_exc()
            return []
    
    def parse_folders(self):
        """Klasör isimlerinden event bilgilerini çıkar
        Format: {EventID}_{YYYYMMDD}_{HHMMSS}_{MagType}_{Mag}/
        Örnek: 659735_20250430_213446_ML_30/
        """
        print(f"\n📂 Klasör isimleri okunuyor: {self.mseed_folders_path}")
        
        if not self.mseed_folders_path or not os.path.exists(self.mseed_folders_path):
            print(f"⚠️ Klasör yolu bulunamadı: {self.mseed_folders_path}")
            return []
        
        # Tüm klasörleri listele
        folders = [f for f in os.listdir(self.mseed_folders_path) 
                   if os.path.isdir(os.path.join(self.mseed_folders_path, f))]
        
        print(f"✅ {len(folders)} klasör bulundu")
        
        # Her klasör ismini parse et
        for folder_name in folders:
            try:
                # Format: {EventID}_{YYYYMMDD}_{HHMMSS}_{MagType}_{Mag}
                parts = folder_name.split('_')
                
                if len(parts) < 5:
                    continue
                
                event_id = parts[0]
                date_str = parts[1]  # YYYYMMDD
                time_str = parts[2]  # HHMMSS
                mag_type = parts[3]  # ML, MW vb
                mag_str = parts[4]   # 30 -> 3.0
                
                # Tarihi parse et
                year = int(date_str[0:4])
                month = int(date_str[4:6])
                day = int(date_str[6:8])
                
                # Saati parse et
                hour = int(time_str[0:2])
                minute = int(time_str[2:4])
                second = int(time_str[4:6])
                
                # Magnitüd değerini düzelt (30 -> 3.0)
                magnitude = float(mag_str) / 10.0
                
                # Datetime oluştur
                event_datetime = datetime(year, month, day, hour, minute, second)
                
                folder_event = {
                    'event_id': event_id,
                    'datetime': event_datetime,
                    'year': year,
                    'month': month,
                    'day': day,
                    'hour': hour,
                    'minute': minute,
                    'second': second,
                    'magnitude': magnitude,
                    'magnitude_type': mag_type,
                    'folder_name': folder_name,
                    'date_string': f"{year:04d}-{month:02d}-{day:02d}"
                }
                
                self.folder_events_info.append(folder_event)
                
            except Exception as e:
                # Hatalı klasör isimlerini atla
                continue
        
        print(f"✅ {len(self.folder_events_info)} klasörden event bilgisi çıkarıldı")
        
        # İlk 5 örneği göster
        if len(self.folder_events_info) > 0:
            print("\n   Örnek klasör eventleri:")
            for i, event in enumerate(self.folder_events_info[:5], 1):
                print(f"     {i}. ID={event['event_id']}, {event['datetime']}, {event['magnitude_type']}={event['magnitude']}")
        
        return self.folder_events_info
    
    def match_sfile_event(self, sfile_event, debug=False):
        """S-file event'ini tarih/saat bazlı eşleştir
        
        Mantık:
        1. ÖNCELİK: Klasör isimlerinden gelen bilgilerle eşleştir (tam tarih-saat eşleşmesi)
        2. YEDEK: Excel bilgileri ile eşleştir (sadece tarih eşleşmesi)
        """
        
        # ÖNCELİK 1: KLASÖR İSİMLERİNDEN TAM ZAMAN EŞLEŞTİRMESİ
        if len(self.folder_events_info) > 0:
            sfile_time = datetime(
                sfile_event['year'],
                sfile_event['month'],
                sfile_event['day'],
                sfile_event['hour'],
                sfile_event['minute'],
                int(sfile_event['second'])
            )
            
            # Klasör eventleri arasında en yakın zamanı bul (±30 saniye tolerans)
            best_folder_match = None
            min_time_diff = float('inf')
            
            for folder_event in self.folder_events_info:
                time_diff = abs((folder_event['datetime'] - sfile_time).total_seconds())
                
                # 30 saniye içinde eşleşme arıyoruz
                if time_diff <= 30 and time_diff < min_time_diff:
                    min_time_diff = time_diff
                    best_folder_match = folder_event
            
            if best_folder_match:
                if debug:
                    print(f"   ✅ KLASÖR EŞLEŞMESI: ID={best_folder_match['event_id']}, Zaman farkı={min_time_diff:.1f}s")

                return best_folder_match
            
            if debug:
                print(f"   ⚠️ Klasörlerde ±30s içinde eşleşme bulunamadı, Excel'e bakılıyor...")
            return None
        
        # ÖNCELİK 2: EXCEL'DEN TARİH BAZLI EŞLEŞTİRME (ESKİ MANTIK)
        # Tarih eşleşmesi için aday event'leri bul (aynı yıl, ay, gün)
        candidates = []
        
        for event_info in self.events_info:
            if (event_info['year'] == sfile_event['year'] and
                event_info['month'] == sfile_event['month'] and
                event_info['day'] == sfile_event['day']):
                candidates.append(event_info)
        
        if len(candidates) == 0:
            if debug:
                search_date = f"{sfile_event['year']}-{sfile_event['month']:02d}-{sfile_event['day']:02d}"
                print(f"   ❌ Excel'de bu tarihte hiç deprem yok: {search_date}")
                
                # En yakın tarihlerdeki depremleri göster
                nearby_dates = []
                for event_info in self.events_info:
                    event_date = f"{event_info['year']}-{event_info['month']:02d}-{event_info['day']:02d}"
                    nearby_dates.append(event_date)
                
                nearby_dates = sorted(set(nearby_dates))
                if nearby_dates:
                    print(f"   💡 Excel'deki en yakın tarihler:")
                    # search_date'e en yakın 3 tarihi bul
                    from datetime import datetime as dt
                    search_dt = dt(sfile_event['year'], sfile_event['month'], sfile_event['day'])
                    
                    date_diffs = []
                    for d in nearby_dates:
                        parts = d.split('-')
                        nearby_dt = dt(int(parts[0]), int(parts[1]), int(parts[2]))
                        diff_days = abs((nearby_dt - search_dt).days)
                        date_diffs.append((d, diff_days))
                    
                    date_diffs.sort(key=lambda x: x[1])
                    for d, diff in date_diffs[:3]:
                        print(f"     - {d} ({diff} gün fark)")
            return None
        
        # DURUM 1: O tarihte TEK deprem varsa direkt eşleştir
        if len(candidates) == 1:
            if debug:
                print(f"   ✅ O tarihte tek deprem var, direkt eşleştiriliyor: ID={candidates[0]['event_id']}")
            return candidates[0]
        
        # DURUM 2: O tarihte BIRDEN FAZLA deprem varsa magnitüd kontrolü yap
        if debug:
            print(f"   ⚠️ O tarihte {len(candidates)} deprem var, magnitüd kontrolü yapılıyor")
        
        # Magnitüd varsa, magnitüd bazlı eşleştir (±0.3 tolerans)
        if sfile_event.get('magnitude') is not None:
            sfile_mag = sfile_event['magnitude']
            
            if debug:
                print(f"   S-file magnitüd: {sfile_mag}")
                for c in candidates:
                    print(f"     Excel: ID={c['event_id']}, Mag={c['magnitude']}, Fark={abs(c['magnitude']-sfile_mag):.2f}")
            
            for candidate in candidates:
                cand_mag = candidate['magnitude']
                mag_diff = abs(cand_mag - sfile_mag)
                
                # Magnitüd değeri yakın olmalı (±0.3)
                if mag_diff <= 0.3:
                    if debug:
                        print(f"   ✅ Magnitüd eşleşti: ID={candidate['event_id']}")
                    return candidate
            
            if debug:
                print(f"   ❌ Hiçbir magnitüd uyuşmuyor (±0.3 tolerans)")
        else:
            if debug:
                print(f"   ⚠️ S-file'da magnitüd yok, en yakın zamanı döndürüyorum")
            
            # Magnitüd yoksa, en yakın zamanı bul
            sfile_time = datetime(
                sfile_event['year'],
                sfile_event['month'],
                sfile_event['day'],
                sfile_event['hour'],
                sfile_event['minute'],
                int(sfile_event['second'])
            )
            
            best_match = None
            min_time_diff = float('inf')
            
            for candidate in candidates:
                time_diff = abs((candidate['datetime'] - sfile_time).total_seconds())
                if time_diff < min_time_diff:
                    min_time_diff = time_diff
                    best_match = candidate
            
            if best_match:
                if debug:
                    print(f"   ✅ En yakın zaman seçildi: ID={best_match['event_id']}, Fark={min_time_diff/60:.1f} dakika")
                return best_match
        
        return None
class SFileParser:
    """Nordic S-file formatını parse eder"""
    
    def __init__(self, sfile_path):
        self.sfile_path = sfile_path
        self.events = []
        
    def parse(self):
        """S-file'ı parse et"""
        print(f"📂 S-file okunuyor: {self.sfile_path}")
        
        with open(self.sfile_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        
        current_event = None
        reading_picks = False
        
        for line_num, line in enumerate(lines, 1):
            line = line.rstrip('\n')
            
            # En az 80 karakter olmalı (son karakter tip göstergesi)
            if len(line) < 80:
                # Boş satır - event sonu
                if current_event is not None and len(line.strip()) == 0:
                    reading_picks = False
                continue
            
            line_type = line[79]
            
            # Yeni event başlangıcı (tip '1' - origin time)
            if line_type == '1':
                # Önceki event'i kaydet
                if current_event is not None:
                    self.events.append(current_event)
                
                # Yeni event
                try:
                    year = int(line[1:5])
                    month = int(line[6:8])
                    day = int(line[8:10])
                    hour = int(line[11:13])
                    minute = int(line[13:15])
                    second = float(line[16:20])
                    
                    origin_time = datetime(year, month, day, hour, minute, int(second))
                    origin_time += timedelta(seconds=second - int(second))
                    
                    # Magnitüd bilgisi (kolon 55-64 arası, örn: "0.3 4.0WAFD")
                    magnitude = None
                    magnitude_type = None
                    try:
                        # Magnitüd string'i al (55-64 arası)
                        mag_full_str = line[55:64].strip()  # örn: "0.3 4.0WAFD"
                        
                        if mag_full_str:
                            # Son kısımı parse et (örn: "4.0WAFD")
                            parts = mag_full_str.split()
                            if len(parts) >= 2:
                                mag_with_type = parts[-1]  # "4.0WAFD"
                            else:
                                mag_with_type = parts[0]
                            
                            # Magnitüd değerini çıkar (sayı kısmı)
                            mag_num_str = ""
                            mag_type_str = ""
                            
                            for i, char in enumerate(mag_with_type):
                                if char.isdigit() or char == '.':
                                    mag_num_str += char
                                else:
                                    mag_type_str = mag_with_type[i:]
                                    break
                            
                            if mag_num_str:
                                magnitude = float(mag_num_str)
                            
                            # Magnitüd tipini belirle (ilk harf)
                            if mag_type_str:
                                first_char = mag_type_str[0].upper()
                                if first_char == 'W':
                                    magnitude_type = 'MW'
                                elif first_char == 'L':
                                    magnitude_type = 'ML'
                                elif first_char == 'B':
                                    magnitude_type = 'Mb'
                                elif first_char == 'S':
                                    magnitude_type = 'Ms'
                                else:
                                    magnitude_type = mag_type_str
                    except:
                        pass
                    
                    current_event = {
                        'origin_time': origin_time.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3],
                        'year': year,
                        'month': month,
                        'day': day,
                        'hour': hour,
                        'minute': minute,
                        'second': second,
                        'magnitude': magnitude,
                        'magnitude_type': magnitude_type,
                        'picks': [],
                        'line_number': line_num,
                        'raw_line': line[:79]
                    }
                    reading_picks = False
                    mag_info = f" ({magnitude_type}={magnitude})" if magnitude and magnitude_type else f" (M={magnitude})" if magnitude else ""
                    # print(f"Event bulundu (satır {line_num}): {origin_time}{mag_info}") # Çok fazla çıktı üretmemesi için kapattım
                except Exception as e:
                    print(f"⚠️ Event parse hatası (satır {line_num}): {e}")
                    print(f"   Satır: {line[:79]}")
                    current_event = None
                    continue
            
            # SPEC satırları (tip '3') - ATLA
            elif line_type == '3':
                continue
            
            # Pick başlık satırı (tip '7' - STAT SP IPHASW...)
            elif line_type == '7':
                if current_event is not None:
                    reading_picks = True
                    # print(f"  Pick başlığı bulundu (satır {line_num})") # Çok fazla çıktı üretmemesi için kapattım
                continue
            
            # Pick satırları (tip ' ' boşluk ve reading_picks aktif)
            elif line_type == ' ' and reading_picks and current_event is not None:
                try:
                    # Station kodu (0-4)
                    station = line[0:5].strip()
                    
                    # Kanal/bileşen (6-8)
                    component = line[6:8].strip()
                    
                    # Faz tipi (10-13) - IP, IS, EP, ES vb.
                    phase_type = line[10:14].strip()
                    
                    # Zaman bilgisi (18-28) - HHMM SS.SS formatında
                    time_str = line[18:28].strip()
                    
                    # DIS (mesafe) - kolon 70-74
                    dis_str = line[70:74].strip() if len(line) > 74 else ""
                    
                    # CAZ (azimuth) - kolon 76-79
                    caz_str = line[76:79].strip() if len(line) > 79 else ""
                    
                    if not station or not phase_type or not time_str:
                        continue
                    
                    # DIS=0.0 veya CAZ=0 ise atla
                    try:
                        if dis_str:
                            dis_val = float(dis_str)
                            if dis_val == 0.0:
                                continue
                        if caz_str:
                            caz_val = int(caz_str)
                            if caz_val == 0:
                                continue
                    except:
                        pass  # Parse edilemezse devam et
                    
                    # Zamanı parse et - HHMM SS.SS formatı
                    parts = time_str.split()
                    if len(parts) >= 2:
                        hhmm = parts[0]
                        seconds = parts[1]
                    else:
                        # Boşluksuz format: HHMMSS.SS
                        if len(time_str) >= 4:
                            hhmm = time_str[:4]
                            seconds = time_str[5:] if len(time_str) > 5 else "0.0"
                        else:
                            continue
                    
                    if len(hhmm) >= 4 and seconds:
                        pick_hour = int(hhmm[0:2])
                        pick_minute = int(hhmm[2:4])
                        pick_second = float(seconds)
                        
                        # Pick time oluştur
                        pick_time = datetime(
                            current_event['year'],
                            current_event['month'],
                            current_event['day'],
                            pick_hour,
                            pick_minute,
                            int(pick_second)
                        )
                        pick_time += timedelta(seconds=pick_second - int(pick_second))
                        
                        # Pick'i kaydet
                        pick = {
                            'station': station,
                            'component': component,
                            'phase': phase_type,
                            'time': pick_time.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3],
                            'line_number': line_num,
                            'raw_time_str': time_str,
                            'raw_line': line[:79]
                        }
                        
                        current_event['picks'].append(pick)
                        # print(f"    Pick: {station} {component} {phase_type} → {pick_time.strftime('%H:%M:%S.%f')[:-3]}") # Çok fazla çıktı üretmemesi için kapattım
                        
                except Exception as e:
                    print(f"⚠️ Pick parse hatası (satır {line_num}): {e}")
                    print(f"   Satır: {line[:79]}")
                    continue
            
            # Event type 'E' - event bilgisi, atla
            elif line_type == 'E':
                continue
        
        # Son event'i kaydet
        if current_event is not None:
            self.events.append(current_event)
        
        print(f"\n✅ Toplam {len(self.events)} event bulundu")
        
        # Her event için pick sayısını göster
        # for idx, event in enumerate(self.events):
        #     print(f"   Event {idx+1}: {event['origin_time']} → {len(event['picks'])} pick")
        
        total_picks = sum(len(e['picks']) for e in self.events)
        print(f"✅ Toplam {total_picks} pick bulundu")
        
        return self.events

In [ ]:
sfile_path = "sfile_27-30D_40-41.5K_01.01.2015_21.05.2025_M3-10.txt"
excel_path = "27-30D_40-41.5K_01.01.2015_21.05.2025_M3-10.xlsx"
mseed_folders_path = "Sismolojik_Veri"
output_json = "sfile_parsed.json"

# --- İşlem Başlangıcı ---
print("="*80)
print("S-FILE PARSING + DUAL EVENT MATCHING (FOLDERS + EXCEL)")
print("="*80)
print()

# 1. Event matcher oluştur ve verileri yükle
matcher = EventMatcher(excel_path, mseed_folders_path)

# 2. Klasör isimlerinden event bilgilerini çıkar (ÖNCELİK!)
folder_events = matcher.parse_folders()
excel_events = matcher.parse_excel()

excel_events_by_id = {}
for excel_event in excel_events:
    if 'event_id' in excel_event:
        excel_events_by_id[excel_event['event_id']] = excel_event

print(f"✅ Excel Lookup oluşturuldu: {len(excel_events_by_id)} event ID'ye göre indekslendi.")

# 4. S-file'ı parse et
parser = SFileParser(sfile_path)
events = parser.parse()

In [ ]:
# 5. Her S-file event'ini klasör/Excel ile eşleştir
print(f"\n{'='*80}")
print("🔄 Event'ler eşleştiriliyor (Önce KLASÖR sonra EXCEL)...")
print(f"{'='*80}\n")

matched_count = 0
matched_by_folder = 0
matched_by_excel = 0
unmatched_events = []

for event in events:
    # 1. S-File event'ini zaman bazlı olarak KLASÖR listesiyle eşleştir
    matched_event = matcher.match_sfile_event(event) 
    
    if matched_event:
        # 2. Eşleşme bulundu (KLASÖR'den). EventID'yi al.
        event_id = matched_event['event_id']
        match_source = "KLASÖR"
        matched_by_folder += 1
        matched_count += 1
        
        event['event_id'] = event_id
        event['match_source'] = match_source

        # --- YENİ EKLENECEK SATIR ---
        # Event'in bulunduğu mseed klasörünün adını da JSON'a ekle
        event['event_folder_name'] = matched_event.get('folder_name')
        # --- YENİ SATIR SONU ---

        # 3. YENİ ADIM: Konum (Location) bilgisini Excel lookup'ından (Hücre 4'te oluşturuldu) al
        if event_id in excel_events_by_id:
            # Excel'de bu ID'ye sahip event'in bilgilerini al
            excel_data = excel_events_by_id[event_id]
            event['location'] = excel_data.get('location') # <-- Konum bilgisi buradan geliyor
            
            # Bonus: Excel'deki Magnitüd/Type'ı da S-file'da eksikse alabiliriz
            if not event.get('magnitude'):
                event['magnitude'] = excel_data.get('magnitude')
            if not event.get('magnitude_type'):
                event['magnitude_type'] = excel_data.get('magnitude_type')
        else:
            # Klasörde bulundu ama Excel listesinde bu ID yoksa
            event['location'] = None 
    
    else:
        # S-File'daki bu event, klasörlerde bulunamadı
        event['event_id'] = None
        event['location'] = None
        event['match_source'] = None
        event['event_folder_name'] = None # Eşleşmeyene de None ekleyelim
        unmatched_events.append(event)

print("✅ Eşleştirme tamamlandı.")

In [ ]:
# Eşleşmeyen event'leri detaylı incele
if len(unmatched_events) > 0:
    print(f"\n{'='*80}")
    print(f"⚠️ EŞLEŞMEYEN {len(unmatched_events)} EVENT DETAYLI ANALİZ (İlk 10'u):")
    print(f"{'='*80}\n")
    
    for i, event in enumerate(unmatched_events[:10], 1):  # İlk 10'unu göster
        print(f"{i}. Event: {event['origin_time']}")
        print(f"   Tarih: {event['year']}-{event['month']:02d}-{event['day']:02d}")
        print(f"   Magnitüd: {event.get('magnitude', 'YOK')}")
        matcher.match_sfile_event(event, debug=True) # Hata ayıklama modunda çalıştır
        print()

print(f"\n{'='*80}")
print("📊 EŞLEŞTİRME İSTATİSTİKLERİ:")
print(f"{'='*80}")
print(f"✅ Toplam eşleşen: {matched_count}/{len(events)}")
print(f"   📁 Klasör isimlerinden: {matched_by_folder}")
print(f"   📊 Excel'den: {matched_by_excel}")
print(f"❌ Eşleşmeyen: {len(unmatched_events)}")
print(f"{'='*80}\n")

# YENİ ADIM: Sadece 'KLASÖR' ile eşleşenleri filtrele
print("Filtreleme: Sadece 'KLASÖR' ile eşleşen event'ler alınıyor...")
filtered_events = [event for event in events if event.get('match_source') == 'KLASÖR']
print(f"S-File'daki {len(events)} event'ten {len(filtered_events)} tanesi klasörlerde bulundu.")

# 5. JSON'a kaydet (FİLTRELENMİŞ LİSTE ile)
print(f"\n📁 JSON'a kaydediliyor: {output_json}")
with open(output_json, 'w', encoding='utf-8') as f:
    # 'events' yerine 'filtered_events' yaz
    json.dump(filtered_events, f, indent=2, ensure_ascii=False) 

# Çıktı mesajını da güncelle
print(f"✅ Tamamlandı! {len(filtered_events)} event {output_json} dosyasına kaydedildi.")

In [ ]:
# Oluşturulan JSON dosyasını Pandas ile okuyup kontrol edelim
try:
    df_sonuc = pd.read_json(output_json)
    
    # 'picks' sütunu liste içerdiği için karmaşık görünebilir,
    # 'picks' olmadan bakalım:
    columns_to_show = ['origin_time', 'event_id', 'match_source', 'magnitude', 'magnitude_type', 'location']
    
    # Sadece ilgili kolonları göster
    display(df_sonuc[columns_to_show].head())
    
    display(df_sonuc['match_source'].value_counts(dropna=False).to_frame())

except Exception as e:
    print(f"JSON okuma hatası: {e}")
    print("Önceki hücrelerin başarıyla çalıştığından emin olun.")